# Xarray-Spatial Morphology

Morphological operators filter rasters by sliding a structuring element (kernel) across the surface and picking the local minimum or maximum at each cell. They show up everywhere from cleaning noisy classification masks to smoothing elevation surfaces before further analysis. This notebook walks through the seven operations in `xrspatial.morphology` on both continuous terrain and binary masks.

### What you'll build

1. Generate synthetic terrain and a hillshade base layer
2. Apply erosion and dilation to see how local min/max reshape the surface
3. Use opening and closing to selectively remove noise
4. Clean up a noisy binary classification mask
5. Compare square and circular structuring elements
6. Use gradient, white top-hat, and black top-hat to extract features

![Morphological operators preview](images/morphological_operators_preview.png)

[Erosion and dilation](#Erosion-and-dilation) · [Opening and closing](#Opening-and-closing) · [Binary mask cleanup](#Binary-mask-cleanup) · [Circular structuring element](#Circular-structuring-element) · [Gradient, white top-hat, black top-hat](#Gradient,-white-top-hat,-and-black-top-hat)

Standard imports plus the morphology submodule.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

import xrspatial
from xrspatial.morphology import (
    _circle_kernel,
    morph_black_tophat,
    morph_closing,
    morph_dilate,
    morph_erode,
    morph_gradient,
    morph_opening,
    morph_white_tophat,
)

## Terrain data

Synthetic elevation built from overlapping Gaussians with added noise. The same raster is reused in every section below.

In [ ]:
W = 800
H = 600
x_range = (-20e6, 20e6)
y_range = (-20e6, 20e6)

terrain = xr.DataArray(np.zeros((H, W)))
terrain = terrain.xrs.generate_terrain(x_range=x_range, y_range=y_range)
illuminated = terrain.xrs.hillshade()

kernel = np.ones((15, 15), dtype=np.uint8)

terrain.plot.imshow(cmap='terrain', size=7.5, aspect=W/H, add_colorbar=False)

Blues are low areas, greens and browns climb to ridges and peaks. The terrain has enough variety to show how morphological operators reshape the surface at different scales.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))
illuminated.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
terrain.plot.imshow(ax=ax, cmap='terrain', alpha=128/255, add_colorbar=False)
ax.set_axis_off()

## Erosion and dilation

[Erosion](https://en.wikipedia.org/wiki/Erosion_(morphology)) replaces each cell with the minimum value inside the kernel footprint. Bright features shrink and valleys widen. [Dilation](https://en.wikipedia.org/wiki/Dilation_(morphology)) does the opposite: it takes the local maximum, so bright features expand and valleys fill in. Both use a 15×15 square kernel here.

The top row shows terrain views on a shared color scale. The bottom row reveals the actual changes: a cross-section profile with shaded fill where the surface moved, and difference maps showing exactly how much each pixel shifted.

In [ ]:
eroded = morph_erode(terrain, kernel=kernel, boundary='nearest')
dilated = morph_dilate(terrain, kernel=kernel, boundary='nearest')

vmin, vmax = float(terrain.min()), float(terrain.max())

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Top row: terrain views
for ax, data, label in zip(
    axes[0],
    [terrain, eroded, dilated],
    ['Original', 'Eroded (local min)', 'Dilated (local max)'],
):
    illuminated.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
    data.plot.imshow(ax=ax, cmap='terrain', alpha=160/255,
                     vmin=vmin, vmax=vmax, add_colorbar=False)
    ax.set_title(label, fontsize=13)
    ax.set_axis_off()

# Bottom left: cross-section with fill-between shading
row = H // 2
cols = np.arange(W)
orig_profile = terrain.values[row]
erode_profile = eroded.values[row]
dilate_profile = dilated.values[row]

ax = axes[1][0]
ax.plot(cols, orig_profile, color='#333', linewidth=1.2, label='Original')
ax.fill_between(cols, orig_profile, erode_profile,
                alpha=0.5, color='#2166ac', label='Eroded away')
ax.fill_between(cols, orig_profile, dilate_profile,
                alpha=0.5, color='#b2182b', label='Dilated gain')
ax.set_title(f'Cross-section at row {row}', fontsize=13)
ax.set_xlabel('Column')
ax.set_ylabel('Elevation')
ax.legend(fontsize=9, loc='upper right')

# Bottom middle and right: difference maps
erode_diff = eroded - terrain
dilate_diff = dilated - terrain
diff_max = max(abs(float(erode_diff.min())), abs(float(dilate_diff.max())))

for ax, diff_data, label in zip(
    axes[1][1:],
    [erode_diff, dilate_diff],
    ['Eroded \u2212 Original', 'Dilated \u2212 Original'],
):
    im = diff_data.plot.imshow(ax=ax, cmap='RdBu_r',
                               vmin=-diff_max, vmax=diff_max,
                               add_colorbar=False)
    ax.set_title(label, fontsize=13)
    ax.set_axis_off()
    fig.colorbar(im, ax=ax, shrink=0.7, label='Elevation change')

plt.tight_layout()

The difference maps make the effect obvious: erosion (blue) pulled every surface downward, while dilation (red) pushed it up. The cross-section shows the same thing quantitatively. Ridgelines, where gradients are steepest, see the largest shifts. Flat areas barely change because the local min and max are close to the original value.

<div class="alert alert-block alert-warning">
<b>Edge handling.</b> The default <code>boundary='nan'</code> treats pixels outside the raster as NaN, which propagates inward by the kernel radius. Use <code>boundary='nearest'</code> (as above) to repeat edge values instead, or <code>'reflect'</code> to mirror them. The choice matters most when your area of interest extends to the raster edge.
</div>

## Opening and closing

[Opening](https://en.wikipedia.org/wiki/Opening_(morphology)) is erosion followed by dilation. It removes small bright features (spikes, noise) while leaving larger structures roughly intact. [Closing](https://en.wikipedia.org/wiki/Closing_(morphology)) is dilation followed by erosion, which fills small dark pits without inflating large features.

Again, the top row shows terrain views and the bottom row shows the actual changes. Opening only subtracts from peaks (blue in the difference map), while closing only adds to valleys (red). Compare this to the erosion and dilation panels above, which shift the entire surface.

In [ ]:
opened = morph_opening(terrain, kernel=kernel, boundary='nearest')
closed = morph_closing(terrain, kernel=kernel, boundary='nearest')

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Top row: terrain views
for ax, data, label in zip(
    axes[0],
    [terrain, opened, closed],
    ['Original', 'Opening (erode \u2192 dilate)', 'Closing (dilate \u2192 erode)'],
):
    illuminated.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
    data.plot.imshow(ax=ax, cmap='terrain', alpha=160/255,
                     vmin=vmin, vmax=vmax, add_colorbar=False)
    ax.set_title(label, fontsize=13)
    ax.set_axis_off()

# Bottom left: cross-section with fill-between shading
row = H // 2
cols = np.arange(W)
orig_profile = terrain.values[row]
open_profile = opened.values[row]
close_profile = closed.values[row]

ax = axes[1][0]
ax.plot(cols, orig_profile, color='#333', linewidth=1.2, label='Original')
ax.fill_between(cols, orig_profile, open_profile,
                alpha=0.5, color='#2166ac', label='Opened (peaks shaved)')
ax.fill_between(cols, orig_profile, close_profile,
                alpha=0.5, color='#b2182b', label='Closed (valleys filled)')
ax.set_title(f'Cross-section at row {row}', fontsize=13)
ax.set_xlabel('Column')
ax.set_ylabel('Elevation')
ax.legend(fontsize=9, loc='upper right')

# Bottom middle and right: difference maps
open_diff = opened - terrain
close_diff = closed - terrain
diff_max = max(abs(float(open_diff.min())), abs(float(close_diff.max())),
               abs(float(open_diff.max())), abs(float(close_diff.min())))

for ax, diff_data, label in zip(
    axes[1][1:],
    [open_diff, close_diff],
    ['Opened \u2212 Original', 'Closed \u2212 Original'],
):
    im = diff_data.plot.imshow(ax=ax, cmap='RdBu_r',
                               vmin=-diff_max, vmax=diff_max,
                               add_colorbar=False)
    ax.set_title(label, fontsize=13)
    ax.set_axis_off()
    fig.colorbar(im, ax=ax, shrink=0.7, label='Elevation change')

plt.tight_layout()

Opening shaved off narrow peaks (blue in the difference map) while closing filled in small depressions (red). Notice the asymmetry: the opening difference is entirely non-positive (it can only remove material), and the closing difference is entirely non-negative (it can only add). The cross-section makes this one-sided behavior easy to see.

## Binary mask cleanup

The most common real-world use of morphological operators is cleaning up classification results. Salt noise (isolated bright pixels in dark regions) and pepper noise (isolated dark pixels in bright regions) are typical artifacts from pixel-level classifiers. Closing fills the pepper holes first, then opening removes the salt specks.

The two panels below show a noisy binary mask before and after a close-then-open pass.

In [ ]:
rng = np.random.default_rng(42)

# Build a binary mask with a large square region
mask = np.zeros((200, 200), dtype=np.float64)
mask[40:160, 40:160] = 1.0

# Sprinkle salt-and-pepper noise
noise = rng.random(mask.shape)
mask[noise < 0.02] = 1.0   # salt (bright specks in dark area)
mask[noise > 0.98] = 0.0   # pepper (dark specks in bright area)

noisy = xr.DataArray(mask, dims=['y', 'x'], name='mask')
mask_kernel = np.ones((5, 5), dtype=np.uint8)
cleaned = morph_opening(
    morph_closing(noisy, kernel=mask_kernel, boundary='nearest'),
    kernel=mask_kernel,
    boundary='nearest',
)

# Color-coded change map (colorblind-safe: blue / orange)
change = cleaned.values - noisy.values
diff_rgb = np.full((*change.shape, 3), 0.85)  # gray: unchanged
diff_rgb[change > 0.5] = [0.12, 0.47, 0.71]   # blue: filled holes (pepper fixed)
diff_rgb[change < -0.5] = [0.89, 0.55, 0.13]  # orange: removed specks (salt fixed)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

noisy.plot.imshow(ax=axes[0], cmap='gray', add_colorbar=False)
axes[0].set_title('Noisy mask', fontsize=13)
axes[0].set_axis_off()

cleaned.plot.imshow(ax=axes[1], cmap='gray', add_colorbar=False)
axes[1].set_title('After closing + opening', fontsize=13)
axes[1].set_axis_off()

axes[2].imshow(diff_rgb)
axes[2].set_title('What changed', fontsize=13)
axes[2].legend(handles=[
    Patch(facecolor='#1f78b4', label='Filled (pepper fixed)'),
    Patch(facecolor='#e38c21', label='Removed (salt fixed)'),
    Patch(facecolor='#d9d9d9', label='Unchanged'),
], loc='lower right', fontsize=10, framealpha=0.9)
axes[2].set_axis_off()

plt.tight_layout()

<div class="alert alert-block alert-warning">
<b>Kernel size vs. feature size.</b> The kernel must be larger than the noise you want to remove. A 3x3 kernel only cleans isolated single-pixel noise. Clusters of 2-3 noisy pixels need a 5x5 or 7x7 kernel. But a larger kernel also eats into legitimate small features, so there is always a trade-off.
</div>

## Circular structuring element

A square kernel introduces directional bias along the diagonals because corner pixels are farther from the center than edge pixels. `_circle_kernel(radius)` builds a disk-shaped structuring element that treats all directions equally, which is usually a better default for natural features.

The top row compares erosion with a 15×15 square kernel against erosion with a circular kernel of radius 7 (giving a 15×15 footprint), plus a binary overlay of where they disagree. The bottom row adds a continuous difference map and a cross-section so you can see the magnitude and spatial pattern of the disagreement.

In [ ]:
disk = _circle_kernel(7)

eroded_square = morph_erode(terrain, kernel=kernel, boundary='nearest')
eroded_disk = morph_erode(terrain, kernel=disk, boundary='nearest')

# Binary disagreement mask
diff = np.abs(eroded_square.values - eroded_disk.values)
diff_mask = xr.DataArray(
    np.where(diff > 0.5, 1.0, np.nan),
    dims=['y', 'x'],
)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Top row: terrain views + binary overlay
illuminated.plot.imshow(ax=axes[0][0], cmap='gray', add_colorbar=False)
eroded_square.plot.imshow(ax=axes[0][0], cmap='terrain', alpha=160/255,
                          vmin=vmin, vmax=vmax, add_colorbar=False)
axes[0][0].set_title('Eroded (15\u00d715 square)', fontsize=13)
axes[0][0].set_axis_off()

illuminated.plot.imshow(ax=axes[0][1], cmap='gray', add_colorbar=False)
eroded_disk.plot.imshow(ax=axes[0][1], cmap='terrain', alpha=160/255,
                        vmin=vmin, vmax=vmax, add_colorbar=False)
axes[0][1].set_title('Eroded (circular r=7)', fontsize=13)
axes[0][1].set_axis_off()

illuminated.plot.imshow(ax=axes[0][2], cmap='gray', add_colorbar=False)
diff_mask.plot.imshow(ax=axes[0][2], cmap=ListedColormap(['darkorange']),
                      alpha=200/255, add_colorbar=False)
axes[0][2].legend(handles=[
    Patch(facecolor='darkorange', alpha=0.78, label='Difference > 0.5'),
], loc='lower right', fontsize=11, framealpha=0.9)
axes[0][2].set_title('Where they disagree', fontsize=13)
axes[0][2].set_axis_off()

# Bottom left: continuous difference map
sq_vs_disk = xr.DataArray(eroded_square.values - eroded_disk.values, dims=['y', 'x'])
sq_dk_max = max(abs(float(sq_vs_disk.min())), abs(float(sq_vs_disk.max())))
im = sq_vs_disk.plot.imshow(ax=axes[1][0], cmap='RdBu_r',
                            vmin=-sq_dk_max, vmax=sq_dk_max,
                            add_colorbar=False)
axes[1][0].set_title('Square \u2212 Circular', fontsize=13)
axes[1][0].set_axis_off()
fig.colorbar(im, ax=axes[1][0], shrink=0.7, label='Elevation difference')

# Bottom middle: cross-section
row = H // 2
ax = axes[1][1]
ax.plot(np.arange(W), eroded_square.values[row],
        label='Square 15\u00d715', color='#2166ac', linewidth=1.5)
ax.plot(np.arange(W), eroded_disk.values[row],
        label='Circular r=7', color='#b2182b', linewidth=1.5)
ax.fill_between(np.arange(W),
                eroded_square.values[row], eroded_disk.values[row],
                alpha=0.3, color='#7570b3')
ax.set_title(f'Cross-section at row {row}', fontsize=13)
ax.set_xlabel('Column')
ax.set_ylabel('Elevation')
ax.legend(fontsize=10)

axes[1][2].set_visible(False)

plt.tight_layout()

The square kernel reaches farther into the diagonal corners than the circular one, so it produces a slightly deeper erosion along ridges that run diagonally. The difference map and cross-section show that the disagreement concentrates on steep slopes where the extra corner pixels pull the minimum lower.

The gradient lights up wherever the terrain has steep transitions, making it a useful edge detector. The white top-hat pulls out peaks and ridges that are narrower than the 15x15 kernel, while the black top-hat does the same for small valleys and depressions. The bottom-right panel (white minus black) is positive on peaks and negative in valleys, giving a signed feature-size map.

All three results are non-negative by construction. The gradient equals the sum of the two top-hats (dilate - erode = (orig - opening) + (closing - orig) when opening <= orig <= closing), which you can verify in the cross-section.

In [ ]:
### References

- [Erosion (morphology)](https://en.wikipedia.org/wiki/Erosion_(morphology)), Wikipedia
- [Dilation (morphology)](https://en.wikipedia.org/wiki/Dilation_(morphology)), Wikipedia
- [Opening (morphology)](https://en.wikipedia.org/wiki/Opening_(morphology)), Wikipedia
- [Closing (morphology)](https://en.wikipedia.org/wiki/Closing_(morphology)), Wikipedia
- [Morphological gradient](https://en.wikipedia.org/wiki/Morphological_gradient), Wikipedia
- [Top-hat transform](https://en.wikipedia.org/wiki/Top-hat_transform), Wikipedia
- [Mathematical morphology](https://en.wikipedia.org/wiki/Mathematical_morphology), Wikipedia
- [Structuring element](https://en.wikipedia.org/wiki/Structuring_element), Wikipedia

## Gradient, white top-hat, and black top-hat

These three operations are derived from the primitives above:

| Operation | Formula | What it extracts |
|-----------|---------|-----------------|
| **Gradient** | dilate - erode | Edges and transitions |
| **White top-hat** | original - opening | Bright features smaller than the kernel |
| **Black top-hat** | closing - original | Dark features smaller than the kernel |

Each is a single subtraction of two morphological results, so all four backends are supported automatically.

In [ ]:
import os, pathlib

# Preview uses difference maps so the four panels look distinct
erode_diff = eroded - terrain
dilate_diff = dilated - terrain
open_diff = opened - terrain
close_diff = closed - terrain
all_diffs = [erode_diff, dilate_diff, open_diff, close_diff]
diff_abs_max = max(max(abs(float(d.min())), abs(float(d.max()))) for d in all_diffs)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, diff_data, label in zip(
    axes,
    all_diffs,
    ['Eroded \u2212 Original', 'Dilated \u2212 Original',
     'Opened \u2212 Original', 'Closed \u2212 Original'],
):
    diff_data.plot.imshow(ax=ax, cmap='RdBu_r',
                          vmin=-diff_abs_max, vmax=diff_abs_max,
                          add_colorbar=False)
    ax.set_title(label, fontsize=11)
    ax.set_axis_off()
plt.tight_layout()

pathlib.Path('images').mkdir(exist_ok=True)
fig.savefig('images/morphological_operators_preview.png',
            bbox_inches='tight', dpi=120)
plt.close(fig)

### References

- [Erosion (morphology)](https://en.wikipedia.org/wiki/Erosion_(morphology)), Wikipedia
- [Dilation (morphology)](https://en.wikipedia.org/wiki/Dilation_(morphology)), Wikipedia
- [Opening (morphology)](https://en.wikipedia.org/wiki/Opening_(morphology)), Wikipedia
- [Closing (morphology)](https://en.wikipedia.org/wiki/Closing_(morphology)), Wikipedia
- [Mathematical morphology](https://en.wikipedia.org/wiki/Mathematical_morphology), Wikipedia
- [Structuring element](https://en.wikipedia.org/wiki/Structuring_element), Wikipedia